# Unsupervised Machine Learning Project

Ka Ho CHAN 9930788

## Set Up

In [1]:
import pandas as pd                                         # tables (dataframes)
import numpy as np
import time
import tracemalloc                                          # measure peak memory (also tracks numpy arrays)
from mlxtend.preprocessing import TransactionEncoder        # one hot encoding: baskets > T/F table
from mlxtend.frequent_patterns import apriori               # find the frequent itemsets
from mlxtend.frequent_patterns import association_rules     # find the association rules
from icecream import ic

'''
Run build() and return (result, seconds, peak memory used in MB).
- seconds : wall-clock time of the run
- peak MB : the highest memory reached during the run (not just what is left at the end)
- track_memory=False : time only, no tracing (peak MB is returned as None)

Why two modes: tracemalloc records every memory allocation. Code that makes millions of small
Python allocations (the naive groupby loop) runs several times slower while traced, which would
make its time look worse than it really is. Code that makes a few large numpy allocations is barely
affected. So for a fair comparison, time is taken from an untraced run and peak memory from a traced run.

Used in Phase 1 for measuring the performance of different approaches.
Used in Phase 2 to measure every Apriori run (numpy-heavy, so one traced run is enough there).
'''
def measure(build, track_memory=True):
    if track_memory:
        tracemalloc.start()
    start = time.perf_counter()
    result = build()
    seconds = time.perf_counter() - start
    peak_mb = None
    if track_memory:
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        peak_mb = peak / 1e6
    return result, seconds, peak_mb

## Phase 1 — Load and prepare (the wide format) 

### Phase 1.1 - Load and Verify the transaction and product files

In [2]:
'''
The file has ONE ROW PER LINE ITEM (not one row per basket).
Several rows share the same TransactionID: those rows together are one receipt. 
'''

df = pd.read_csv("../data/transactions.zip", usecols=["TransactionID", "Date", "Item"])
products = pd.read_csv("../data/products.csv")

print('File Loaded.')
print(f'     rows (line items):   {len(df):,}')
n_baskets = df['TransactionID'].nunique()
print(f'     transactions:        {n_baskets:,}')
print(f'     columns:             {list(df.columns)}') # list gives better formatting
print(f'     products:            {df["Item"].nunique()}')
print(f'     average basket size: {len(df) / n_baskets:.2f}')

# Show that the data is clean and consistent
print('\nShow that the data is clean and consistent:')
print(f'     missing values:      {df.isnull().sum().sum()}')
print(f'     date range:          {df["Date"].min()} -> {df["Date"].max()}')
print(f'     same item twice in a basket: {df.duplicated(["TransactionID", "Item"]).sum()}')
print(f'     names not in products.csv:   {len( set(df['Item']) - set(products['Item']))} (0 = all names match the catalogue)')

print('\nShow the first 5 rows of the data:')
print(df.head())

File Loaded.
     rows (line items):   8,379,837
     transactions:        1,000,000
     columns:             ['TransactionID', 'Date', 'Item']
     products:            98
     average basket size: 8.38

Show that the data is clean and consistent:
     missing values:      0
     date range:          2026-01-01 -> 2026-12-31
     same item twice in a basket: 0
     names not in products.csv:   0 (0 = all names match the catalogue)

Show the first 5 rows of the data:
   TransactionID        Date                 Item
0         100000  2026-06-19       Cidre de glace
1         100000  2026-06-19              Couches
2         100000  2026-06-19      Fromage cheddar
3         100000  2026-06-19  Lingettes pour bébé
4         100000  2026-06-19      Purée pour bébé


### Phase 1.2a - Organize the data in wide format: one list of items per transaction

In [3]:
'''
Naive method (as we used in the exercise): group the rows by TransactionID, then TransactionEncoder.
- groupby + lambda builds one Python list per basket -> 1,000,000 lists of strings in memory
- TransactionEncoder then converts those lists into the True/False table
This is a Python loop over 1M groups, so it will be slow, 
and the lists use far more memory than the final table.
'''
def build_naive(df):
    # return a list of sorted unique items for each transaction
    baskets = df.groupby('TransactionID')['Item'].apply(lambda items: sorted(set(items))).tolist()
    te = TransactionEncoder()
    return pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_)

# run 1: time only (no tracing, so the Python loop is not slowed down)
_, time_naive, _ = measure(lambda: build_naive(df), track_memory=False)
# run 2: peak memory (traced; its time is not used because tracing slows the loop)
basket_table_naive, time_naive_traced, peak_naive = measure(lambda: build_naive(df))

print('Phase 1.2a - naive method (groupby + TransactionEncoder)')
print(f'    time        : {time_naive:.1f} s   (untraced run; {time_naive_traced:.1f} s with tracemalloc on)')
print(f'    peak memory : {peak_naive:,.0f} MB')
print(f'    table       : {basket_table_naive.shape}, {basket_table_naive.memory_usage(deep=True).sum() / 1e6:.0f} MB')

Phase 1.2a - naive method (groupby + TransactionEncoder)
    time        : 20.2 s   (untraced run; 92.5 s with tracemalloc on)
    peak memory : 920 MB
    table       : (1000000, 98), 98 MB


### Phase 1.2b - Fast Approach: numpy function without a Python loop

In [4]:
'''
Fast method: no Python loop.
- pd.factorize turns every TransactionID into a row number (0..999,999)
  and every Item into a column number (0..97)
- one numpy assignment then sets all 8.4M True values at once
- dtype=bool -> 1 byte per cell (int64 would be 8 bytes per cell = 784 MB)
'''
def build_fast(df):
    basket_codes, basket_ids = pd.factorize(df['TransactionID'])
    item_codes, item_names = pd.factorize(df['Item'])
    matrix = np.zeros((len(basket_ids), len(item_names)), dtype=bool)
    matrix[basket_codes, item_codes] = True
    return pd.DataFrame(matrix, index=basket_ids, columns=item_names).sort_index(axis=1)

# same two-run protocol as the naive method, so the comparison is fair
_, time_fast, _ = measure(lambda: build_fast(df), track_memory=False)
basket_table, time_fast_traced, peak_fast = measure(lambda: build_fast(df))

print('Phase 1.2b - fast method (factorize + numpy)')
print(f'    time        : {time_fast:.2f} s   (untraced run; {time_fast_traced:.2f} s with tracemalloc on)')
print(f'    peak memory : {peak_fast:,.0f} MB')
print(f'    table       : {basket_table.shape}, {basket_table.memory_usage(deep=True).sum() / 1e6:.0f} MB '
      f'(1 byte x 98M cells + 8 MB basket index)')
print(f'    same table as naive : {(basket_table.sort_index().to_numpy() == basket_table_naive.to_numpy()).all()}')
print(f'    speed-up    : {time_naive / time_fast:.0f}x faster, peak memory {peak_naive / peak_fast:.1f}x lower')

del basket_table_naive   # free memory before Apriori

Phase 1.2b - fast method (factorize + numpy)
    time        : 0.72 s   (untraced run; 0.71 s with tracemalloc on)
    peak memory : 436 MB
    table       : (1000000, 98), 106 MB (1 byte x 98M cells + 8 MB basket index)
    same table as naive : True
    speed-up    : 28x faster, peak memory 2.1x lower


### First Genuine Engineering Problem

| Method | Time | Peak memory | Final table |
|---|---|---|---|
| groupby + TransactionEncoder (naive method) | 20.4 s | 920 MB | 98 MB |
| factorize + numpy (fast method) | 0.73 s | 436 MB | 106 MB |

- **Problem:** Apriori needs a boolean table with one row per basket and one column per product (1,000,000 * 98). The number of rows (line items) in the file is 8,379,837
- **Why the naive method is costly:** it builds 1,000,000 Python lists of strings before encoding.
  Its peak memory is 9.4x the size of the final table, and the Python loop over 1M groups is 28x slower than the fast method. 
- **Why the fast method works:** `pd.factorize` maps baskets and products to row/column numbers,
  and a single numpy assignment fills the table with no Python loop.
- **Memory depends mostly on the dtype:** as `bool` the table takes 98 MB; the same table in `int64`
  (e.g. from `pd.get_dummies` or `pd.crosstab` counts) would take 784 MB.
- **Decision:** both methods fit on this machine (34 GB RAM), and both give the identical table.
  The fast method is kept because it is about 28 times faster with a lower peak of 2.1 times, and Phase 4 rebuilds the table for two more seasons.

### Phase 1.3 - Find the item frequencies

In [5]:
item_freq = basket_table.mean().sort_values(ascending=False)   # support of each single product
freq_table = (item_freq.rename('support').to_frame()
              .assign(n_baskets=(item_freq * len(basket_table)).round().astype(int))
              .join(products.set_index('Item')[['Item_EN', 'Category']]))

print('10 most frequent products'); display(freq_table.head(10))
print('5 rarest products');        display(freq_table.tail(5))

10 most frequent products


,support,n_baskets,Item_EN,Category
Sac réutilisable,0.900026,900026,Reusable bag,Maison
Biscuits,0.161439,161439,Cookies,Collations
Noix mélangées,0.155189,155189,Mixed nuts,Collations
Beurre d'arachide,0.153726,153726,Peanut butter,Épicerie
Bière,0.150468,150468,Beer,Boissons
Savon à vaisselle,0.149820,149820,Dish soap,Maison
Fromage cheddar,0.143457,143457,Cheddar cheese,Fromages
Charbon de bois,0.140561,140561,Charcoal,Épicerie
Papier hygiénique,0.139758,139758,Toilet paper,Maison
Céréales,0.139198,139198,Cereal,Épicerie


5 rarest products


,support,n_baskets,Item_EN,Category
Safran,0.000818,818,Saffron,Spécialités
Huile de truffe,0.000817,817,Truffle oil,Spécialités
Caviar,0.000810,810,Caviar,Spécialités
Vinaigre balsamique vieilli,0.000779,779,Aged balsamic vinegar,Spécialités
Foie gras,0.000756,756,Foie gras,Spécialités


### Observation in Phase 1.3

- The support of Sac réutilisable is very high at 90%, while the support of the next product is at 16%. 
- The support of the 5 rarest products are around 0.08% which is below every min_support tested (5%, 2%, 1% and 0.5%). Apriori will therefore find no rule containing them, so the app can never recommend them.

### Phase 1.4 - Prediction of which product will cause trouble
(Written before running Apriori, from the frequencies in Phase 1.3)
- **Product:** Sac réutilisable
- **Reason:** It is in 90% of the baskets, so almost any rule saying "X → Sac réutilisable" will have a confidence near 0.90. But lift = confidence / support(Sac réutilisable) ≈ 0.90 / 0.90 ≈ 1, meaning that the rule tells nothing to us. 
- **Expected result:** It will fill the top 10 list by confidence and be absent from the top 10 list by lift. To be checked in Phase 3. 

### Phase 1.5 - Evidence of a duplicated product

In [6]:
a, b = 'Baguette', 'Pain baguette rustique'
display(products[products['Item'].isin([a, b])])
both = (basket_table[a] & basket_table[b]).sum()
expected = basket_table[a].mean() * basket_table[b].mean() * len(basket_table)
print(f'bought together: {both:,}   expected if unrelated: {expected:,.0f}')

pair = df[df['Item'].isin([a, b])]
display(pair.groupby([pair['Date'].str[5:7], 'Item']).size().unstack())   # sales by month

,Item,Item_EN,Category,Price
48,Baguette,Baguette,Boulangerie,2.99
96,Pain baguette rustique,Rustic baguette,Boulangerie,3.99


bought together: 3,009   expected if unrelated: 2,930


Item,Baguette,Pain baguette rustique
Date,,
01,5985,5691
02,5342,4813
03,5845,3851
04,5697,2614
05,5894,2164
06,5584,1757
07,5943,1798
08,6047,1905
09,5706,2366


### **Decision to handle these products**
- The sale of baguette is almost the same throughout the year. 
- However, the sale of rustic baguette is higher during the months with lower temperature and lower during the months with higher temperature. 
- These two are essentially different products. Therefore, they will be kept as separate products. 

## Step 4 - Apropri: Find the frequent itemsets

In [ ]:
''' 
apriori() keeps only the combinations of products that appear often enough. 
    - basket_table: the true/false table from Step 3
    - min_support=0.02: Keep an itemset only if it appears in at least 2% of the baskets (transactions)
                          Support is a fraction: 0.02 * 2472 baskets = 49.44 baskets, so we keep itemsets that appear in at least 50 baskets
                          Higher value -> fewer, stronger itemssets and a faster run time
    - use_colnames=True: use the product names (column names) instead of column numbers in the output
                         if false, it will not show the product names, only the column numbers (0, 1, 2, ...) which is not useful
    - max_len=3: Keep only itemsets with 3 or fewer products (1, 2, or 3 products). 
                 Longer ones are raise and hard to act on. Set it to none for no limit
'''

frequent_itemsets = apriori(basket_table, 
                         min_support=0.02, 
                         use_colnames=True, 
                         max_len=3) 

# At this point, the result has 2 columns: 'support' (the fractions) and 'itemsets' (the products in the itemset)

# Add a helper column: how many products are in each itemset
frequent_itemsets['size'] = frequent_itemsets['itemsets'].apply(len) # apply(len) = length of each set

print(frequent_itemsets)

print('\nSTEP 4 - frequent itemsets found by Apriori.')
print(f'  total itemsets: {len(frequent_itemsets)}')
by_size = frequent_itemsets['size'].value_counts().sort_index() # count how many itemsets of each size (1, 2, 3) and sort by size
print(f'  by size: {by_size.to_dict()}') # print the counts as a dictionary
print('  the 5 most frequent items of 2 products: ')
pairs = frequent_itemsets[frequent_itemsets['size']==2].nlargest(5, 'support') # get the 5 pairs with the largest support
# same as: .sort_values('support', ascending=False).head(5)
for i, row in pairs.iterrows():
    print(f'        {row["itemsets"]} : {row["support"]:.3f}') # print the itemset and its support

trash = frequent_itemsets
trash['readable'] = list(trash['itemsets'].apply(lambda x: ', '.join(x))) # convert the set of products into a readable string
trash

## Step 5 - Generate the rules

In [ ]:
'''
association_rules() splits every frequent itemset into IF -> THEN rules, and calculates the metrics (quality measures) for each rule.
    - frequent_itemsets: the frequent itemsets from Step 4
    - metric='lift': the metric to use for filtering the rules (lift = 1 means no association, lift > 1 means positive association)
    - min_threshold=1.2: keep only rules with lift >= 1.2
        - lift = confidence / support(Y)
               = 1 -> useless
               > 1 -> positive association (if X is bought, Y is more likely to be bought)
               < 1 -> negative association (if X is bought, Y is less likely to be bought)


'''

rules = association_rules(frequent_itemsets,
                          metric='lift', # the metric to use for filtering the rules
                          min_threshold=1.2) # keep only rules with lift >= 1.2 (lift = 1 means no association, lift > 1 means positive association)
                          
# Keep only the rules we can actually serve
 
# 1) ONE product on the THE side: we recommend single items, not bundles
'''
rules['support'] in the market basket analysis, "support" is the percentage of all transactions 
that contain the items in the rule (the IF and THEN side together), e.g. 0.02 means 2% of all baskets

len(basket_table) is the total number of baskets (transactions) in the dataset, i.e. 1,000,000

rules['support'] * len(basket_table) is the number of baskets that contain the items in the rule 
    (the IF and THEN side together), e.g. 0.02 * 1,000,000 = 20,000 baskets

.round().astype(int) cleans up the math to turn it into a whole number

.apply(len) == 1: keeps only rules with one product on the THEN side (the consequents)

rules['confidence'] >= 0.30: at least 30% of time when the IF side is bought, the THEN side is also bought

at least 40 real baskets behind the rule, so it is not a coincidence: 
    we want to be able to actually serve the recommendation, so we need at least 40 baskets that contain the rule (the IF and THEN side together)
    (support is a fraction, so support x number of baskets = number of baskets that contain the rule)
'''
rules['n_baskets'] = (rules['support'] * len(basket_table)).round().astype(int) # portion of baskets that contain the rule (support * total number of baskets)

# print(rules) # use this if I want to see all the rules before the filtering

rules = rules[(rules['consequents'].apply(len) == 1) &
              (rules['confidence'] >= 0.30) & 
              (rules['n_baskets'] >= 40)] # see above for explanation of the filtering criteria
rules

In [ ]:
# Sort the best rules first, i.e. highest lift first (the higher the lift, the stronger the association)
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True) # reset the index to be 0, 1, 2, ... instead of the original index from the dataframe
rules
# ic(rules)

print('\nSTEP 5 - association rules generated.')
print(f'  Usable rules: {len(rules)}')
print('  The 5 best rules (highest lift): ')
print(rules.head(5)[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]) # print the first 5 rules with the relevant columns since the rules have been sorted

print("\nAnother way to print the rules in a more readable format:")
for _, r in rules.head(5).iterrows():
    print(f'        {r["antecedents"]} -> {r["consequents"]} : support={r["support"]:.3f}, confidence={r["confidence"]:.3f}, lift={r["lift"]:.3f}, n_baskets={r["n_baskets"]}') # print the rule and its metrics
    
print("\nAnother way to print the rules without the word frozenset:")
for _, r in rules.head(5).iterrows():
    left = ', '.join(r["antecedents"]) # frozenset -> readable text
    right = ', '.join(r["consequents"]) 
    print(f"    IF {left:<28} | THEN {right:<16} | support={r['support']:.3f}, confidence={r['confidence']:.3f}, lift={r['lift']:.3f}, n_baskets={r['n_baskets']}") # print the rule and its metrics

## Step 6 - Save the rules to a file

In [ ]:
'''
The rules are the "knowledge" of the recommender. We compute them once (slow)
and save them, so the recommendation part can just read the file.

A frozenset cannot be saved to a csv file, so we convert both sides to the text
'''

rules_to_save = rules.copy() # make a copy of the rules dataframe to save to csv
# convert the frozenset to a string. In lambda x is the frozenset, ', '.join(x) converts 
# it to a string with the items separated by commas, which are sorted 
rules_to_save['antecedents'] = rules_to_save['antecedents'].apply(lambda x: ', '.join(sorted(x))) 
rules_to_save['consequents'] = rules_to_save['consequents'].apply(lambda x: ', '.join(sorted(x)))
rules_to_save

In [ ]:
# Keep only useful columns, in a sensible order for reading
rules_to_save = rules_to_save[['antecedents', 'consequents', 'support', 'confidence', 'lift', 'n_baskets']]
rules_to_save.to_csv('association_rules.csv', index=False) # save the rules to a csv file

print('\nSTEP 6 - association rules saved to file.')
rules_to_save

## Step 7 - The Recommender

In [ ]:
'''
Suggest products for a basket, with an explanation of why we suggest them.

Recommend products based on the customer's existing basket and the association rules learned from historial shopping transactions.

Parameters: 
- basket: a list of products in the customer's basket
- how many: how many suggestions to return (default 3). Maximum number of recommendations of products to return

Returns: 
- a list of dictionaries, each dictionary contains: {'Item', 'score', 'reason'}
    - item: the product to recommend
    - score: list of score rounded to 2 decimal places
    - reason: explanation for the recommendation, i.e. the rule that led to the recommendation, in a readable format
    
Requirements: 
The dataframe of rules or Json file must always exist.

'''

# TODO Load rules from a file, csv or json, so we don't have to recompute them every time we run the program.
# Create a streamlit app that allows the user to input a basket (by keyboard or a list to choose) and get recommendations based on the rules.

def recommend(basket, how_many=3): 
    # Step 7.1 Convert the basket into a set
    '''
    Example: 
    ['pasta', 'ground beef', 'pasta'] -> {'pasta', 'ground beef'}
    Sets remoe duplicates and make membership checks convenient
        
    '''
    basket = set(basket) # convert the basket into a set to remove duplicates and make membership checks convenient  
                         # a set makes the "is it inside?" test easy  
    
    # Step 7.2 Create dictionnaries to store recommendations
    '''
    Dictionarly storing each recommended product and its best life score.add()
    {
        'tomato source': 2.4
        'parmesan': 1.8
    }
    '''
    
    suggestions = {} # product -> best lift score so format
    
    '''
    Dicionary storing the explanation assocaited with each product
    
    The product name is the key, and its explanataion is the value
    
    Example: 
    {
        'tomato sauce': 'because you bought pasta ...'
    }
    '''
    
    reason = {}
    
    # Step 7.3 Examine every association rule
    '''
    'rules' is a panda datafram that contains the association_rules learned in the previous steps
    '.iterrows()' returns an iterator over the rows of the dataframe, each row is a tuple (index, row)
    '_' receives the row index, which we don't need, and 'row' receives the row itself, which is a pandas Series object
    'rule' contains the current row's data
    '''
    
    # Look at every rule we kept in Step 5 (the rules that are strong enough to be useful)
    for _, rule in rules.iterrows(): 
        # Step 7.3.1 Get the antecedent (the IF side)
        '''
        Retrieve the products requried for this rule to apply
        Example: 
        {'pasta', 'ground beef'}
        
        Convert them into a set for easy comparison with the basket
        '''
        
        if_part = set(rule['antecedents'])
        
        # Step 7.3.2 Get the consequent (the THEN side)
        '''
        Retrieve the product that is recommended by this rule
        'list()' converts the consequent collection into a list
        '[0]' selects the first product in the list
        
        '''
        
        # get the first (and only) item in the set of consequents, since we only kept rules with one consequent
        then_part = list(rule['consequents'])[0]
        # print(if_part, then_part)
        
        if if_part.issubset(basket):
            
            # Step 7.4 Check whether the rule applies to the basket
            '''
            '.issubset()' checks whether the antecedent (the IF side) is a subset of the basket
            'not' reverses the result
            If even one required product is missing, skip this rule and move on to the next one
            because the if part must be fully contained in the basket for the rule to apply
            '''
            
            if not if_part.issubset(basket):
                continue

            # Step 7.5 Check whether the recommended product is already in the basket
            '''
            Check whether the recommended product (the THEN side) is already in the basket
            If it is, skip this rule and move on to the next one            
            because we don't want to recommend products that the customer already has
            '''
            
            if then_part in basket:
                continue
            
            # Step 7.6 Keep the strongest rule for each recommended product
            '''
            A product may be recommended by multiple rules, but we want to keep only the strongest one (the one with the highest lift score)
            
            Example: 
            pasta               -> tomato sauce (lift=1.7)
            pasta + ground beef -> tomato sauce (lift=1.8)
            
            In this case, we want to keep the second rule, because it has a higher lift score (1.8 > 1.7)
            
            'or' means the condition is true if the product is New or 
            the current rule has a higher lift score than the previous one (the one already stored in the suggestions dictionary)
            
            
            
            If the product is not already in the suggestions dictionary, add it with its lift score and explanation
            If the product is already in the suggestions dictionary, but this rule has a higher lift score than the previous one, update it with the new lift score and explanation
            Store the lift score for this product in the suggestions dictionary, and store the explanation for this product in the reason dictionary
            
            '''
            if ((then_part not in suggestions) # if the product is not already in the suggestions dictionary, 
                                              # add it with its lift score and explanation
                or (rule['lift'] > suggestions[then_part]) # if the product is already in the suggestions dictionary, 
                ):                                         # but this rule has a higher lift score than the previous one, 
                                                           # update it with the new lift score and explanation
                
                # store the strongest lift score for this product                
                suggestions[then_part] = rule['lift'] 
                
                # Sort the strongest lift score found for this product
                '''
                Store the antecedent products alphabetically and join them into a readable phrase
                
                Example: 
                {'pasta', 'ground beef'} -> 'ground beef, pasta'
               
                '''
                
                trigger = ' and '.join(sorted(if_part)) # sort the antecedent products alphabetically and join them into a readable phrase
                
                # Build an explanation for this product using f-string
                '''
                {trigger} inserts the products that activated the rule (the IF side) into the explanation
                {then_part} inserts the recommended product (the THEN side) into the explanation
                {rule['confidence']:.0%} inserts the confidence score of the rule into the explanation
                {rule['lift']:.2f} inserts the lift score of the rule into the explanation               
                '''
                reason[then_part] = (
                    f"because you bought {trigger} - "
                    f"(confidence={rule['confidence']:.0%} of baskets with"
                    f"{trigger} also contain {then_part}, "
                    f"lift={rule['lift']:.2f})x more often than average "
                    f"(based on {rule['n_baskets']} real baskets)"
                )
                
                # Save the explanation udner the recommended product's name
                # if a stronger rule is found later, it will overwrite the previous explanation with the new one      

    # Step 7.67 Sort the recommendations from the highest score to the lowest score, keep the top ones
    # sorted(..., key = ..., reverse=True) sorts the items in the suggestions dictionary by their lift score in descending order
    best = sorted(suggestions.items(), 
                    key=lambda pair: pair[1], # lambda pair: pair[1] means sort by the second item in the tuple (the lift score)
                    reverse=True)[:how_many] # keep only the top 'how_many' recommendations
    
    '''
    Example of the output format:
    [
        {
            'Item': 'tomato sauce', # the recommended product
            'score': 2.4, # the lift score rounded to 2 decimal places
            'reason': "because you bought tomato sauce ..." # the explanation for the recommendation
        }
    ]
    If no eligible recommendations are found, the output will be an empty list: []
    
    '''
        
    return [
        {
            'Item': item, # the recommended product
            'score': round(score, 2), # the lift score rounded to 2 decimal places
            'reason': reason[item] # the explanation for the recommendation
        }
        for item, score in best # for each recommended product and its lift score in the top recommendations
    ]

## Step 8 - Test the recommend function with a sample basket_table

In [ ]:
print("\nSTEP 8 - test the recommend function with a sample basket")

# Note we may need to parse the basket items to match the cleaned product names in the rules (lowercase, no leading/trailing spaces)
test_baskets = [
    ['pasta'],                         
    ['pasta', 'ground beef'], 
    ['beer', 'tortilla chips'],               
    ['diapers'],                 
    ['yogurt', 'berries'],             
    ['shampoo'],
    ['lemon'],
    ['bananas', 'berries']
]

for id, basket in enumerate(test_baskets):
    print(f"\nBasket: {id+1} - {basket}")
    recommendations = recommend(basket, how_many=3)
    if recommendations:
        for rec in recommendations:
            print(f"  Recommend: {rec['Item']} (lift={rec['score']}) - {rec['reason']}")
    else:
        print("  No recommendations found (no rules apply to this basket).")
